# Volatility Filter (ATR Rising = No Entry) on SPY
## Strategy Brief
This strategy uses the Average True Range (ATR) as a volatility filter to avoid entering trades when volatility is rising. The hypothesis is that rising volatility may indicate uncertainty or potential reversals, making it a less favorable time to enter new positions. The strategy avoids entering trades when the ATR is rising, and only considers entering when the ATR is stable or falling. This approach aims to improve risk-adjusted returns by avoiding volatile periods that may lead to larger drawdowns.
## References
- https://www.investopedia.com/terms/v/volatility.asp

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
Define the trading context and parameters for the strategy.

In [ ]:
ATR_PERIOD = 14
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SYMBOL = 'SPY'

### PHASE 2 - Data Exploration
Download SPY data using yfinance, compute ATR, and plot it overlaid on the price.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(SYMBOL, start=START_DATE, end=END_DATE)

# Compute ATR
high_low = data['High'] - data['Low']
high_close = np.abs(data['High'] - data['Close'].shift())
low_close = np.abs(data['Low'] - data['Close'].shift())
true_range = np.maximum(high_low, high_close, low_close)
data['ATR'] = true_range.rolling(window=ATR_PERIOD).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(data['ATR'], label='ATR', linestyle='--')
plt.title('SPY Price and ATR')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
Create the signal based on ATR and define entry/exit logic.

In [ ]:
# Signal: ATR rising = no entry
atr_rising = data['ATR'] > data['ATR'].shift(1)

# Entry/Exit logic
positions = pd.Series(index=data.index, data=0)
positions[~atr_rising] = 1  # Long when ATR is not rising

# Forward fill positions to maintain them
positions = positions.ffill().fillna(0)

### PHASE 4 - Coding & Backtesting
Calculate daily returns, apply positions, and plot the equity curve.

In [ ]:
# Shift positions to align with next day's returns
shifted_positions = positions.shift(1)

# Calculate daily returns
daily_returns = data['Close'].pct_change()
strategy_returns = shifted_positions * daily_returns

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Strategy Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
Evaluate the performance using CAGR, Sharpe, Sortino, Calmar ratios, and compare with buy-and-hold.

In [ ]:
def calculate_performance(equity_curve):
    total_return = equity_curve.iloc[-1] - 1
    n_years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (equity_curve.iloc[-1]) ** (1 / n_years) - 1
    
    # Calculate drawdown
    rolling_max = equity_curve.cummax()
    drawdown = (equity_curve - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    # Calculate Sharpe and Sortino
    daily_volatility = strategy_returns.std()
    sharpe_ratio = strategy_returns.mean() / daily_volatility * np.sqrt(252)
    downside_volatility = strategy_returns[strategy_returns < 0].std()
    sortino_ratio = strategy_returns.mean() / downside_volatility * np.sqrt(252)
    
    # Calmar ratio
    calmar_ratio = cagr / abs(max_drawdown)
    
    return {
        'CAGR': cagr,
        'Max Drawdown': max_drawdown,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio
    }

# Calculate performance
strategy_performance = calculate_performance(equity_curve)

# Buy-and-hold performance
equity_bh = (1 + daily_returns).cumprod()
bh_performance = calculate_performance(equity_bh)

# Comparison table
performance_df = pd.DataFrame([strategy_performance, bh_performance], index=['Strategy', 'Buy & Hold'])
print(performance_df)

### PHASE 6 - Deploy & Monitor
Create a function to download recent data and compute today's signal.

In [ ]:
def get_recent_signal(symbol, atr_period=ATR_PERIOD):
    recent_data = yf.download(symbol, period='60d')
    
    # Compute ATR
    high_low = recent_data['High'] - recent_data['Low']
    high_close = np.abs(recent_data['High'] - recent_data['Close'].shift())
    low_close = np.abs(recent_data['Low'] - recent_data['Close'].shift())
    true_range = np.maximum(high_low, high_close, low_close)
    recent_data['ATR'] = true_range.rolling(window=atr_period).mean()
    
    # Signal: ATR rising = no entry
    atr_rising = recent_data['ATR'] > recent_data['ATR'].shift(1)
    
    # Today's position
    if atr_rising.iloc[-1]:
        print(f"Today's signal for {symbol}: No Entry (ATR Rising)")
    else:
        print(f"Today's signal for {symbol}: Long Entry (ATR Not Rising)")

# Get today's signal
get_recent_signal(SYMBOL)